# 260817 cindysha 
### 1) `.gitignore`
**文件作用**：定义 Git 忽略规则。  
**本次改动**：
- 新增忽略 `.DS_Store`
- 新增忽略 `CODEBUDDY.md`
- `AGENT.md` 行尾调整（本质规则不变）

**改后功能**：
- 避免 macOS 系统文件、开发辅助文档被误提交，仓库更干净。

---

### 2) `grc/agent-backend/__init__.py`（删除）
**文件作用（原）**：旧的 `agent-backend` 包入口。  
**本次改动**：整文件删除。  
**改后功能**：
- 表示包结构迁移：从 `grc/agent-backend` 迁到 `grc/agent`（见下方新增/重命名文件）。

---

### 3) `grc/agent/__init__.py`（新增，核心）
**文件作用**：新的 Agent 高层入口。  
**本次改动**：
- 新增 `build_flow_graph_from_text(...)` 主入口：
  - 接收自然语言需求
  - 生成 FlowGraph
  - `rewrite + validate`
  - 保存为 `.grc` 并返回路径
- 新增两条生成路径：
  1. **LLM 路径**：配置了 `GRC_AGENT_*` 时，调用 `grc.agent.llm` 直接生成 `.grc YAML` 后导入
  2. **回退路径**：未配置 LLM 时生成内置 demo 流图
- 新增 YAML 解析与导入 `_flow_graph_from_grc_text`
- 新增 demo 组图 `_demo_flow_graph`

**改后功能**：
- GRC 具备“文本意图 → 自动建图 → 输出/打开 .grc”的统一后端能力。
- 即使没配 LLM，也能跑通端到端主链路（保证可用性）。

---

### 4) `grc/agent-backend/env.py` → `grc/agent/env.py`（重命名，内容未变）
**文件作用**：Agent 环境桥接/平台构建辅助。  
**本次改动**：纯路径迁移。  
**改后功能**：
- 与新包命名 `grc.agent` 对齐，便于统一导入。

---

### 5) `grc/agent-backend/examples/__init__.py` → `grc/agent/examples/__init__.py`（重命名）
**作用**：examples 子包声明。  
**改后功能**：路径归一到新包。

---

### 6) `grc/agent-backend/examples/bpsk_2g4_regression.py` → `grc/agent/examples/bpsk_2g4_regression.py`（重命名）
**作用**：示例/回归脚本。  
**改后功能**：迁移到新目录，逻辑不变。

---

### 7) `grc/agent/llm.py`（新增，核心）
**文件作用**：LLM 接入层（OpenAI-compatible Chat Completions）。  
**本次改动**：
- 新增配置读取：`GRC_AGENT_BASE_URL / API_KEY / MODEL / TIMEOUT / MAX_MESSAGES`
- 新增配置检查 `is_configured()`
- 新增系统提示词 `SYSTEM_PROMPT`（约束模型输出合法 `.grc YAML`）
- 新增消息组装（支持 history 截断）
- 新增 HTTP 调用（`urllib`，无第三方依赖）
- 新增健壮错误处理（HTTP/JSON/字段缺失/空响应）
- 新增输出后处理：去掉 ```yaml 围栏，提取纯 YAML

**改后功能**：
- Agent 可以直接通过外部大模型“按约束生成 GRC 流图文本”；
- 且具备配置化、可控超时、历史多轮上下文与容错。

---

### 8) `grc/core/blocks/options.py`
**文件作用**：GRC `options` 块与 workflow 选择逻辑。  
**本次改动**（`update_current_workflow`）：
- 从 `get_value()` 改为读参数原始值 `.value` 来匹配 workflow
- 更新断言报错信息中的取值来源

**改后功能**：
- 修复初始化早期阶段 workflow 匹配失真问题（尤其 `output_language` / `generate_options` 在首轮 rewrite 时被错误清洗导致匹配失败）。
- 直接收益：不会错误把有效语言（如 python）重置为空，workflow 更稳定。

---

### 9) `grc/core/default_flow_graph.grc`
**文件作用**：默认新建流图模板。  
**本次改动**：
- 在 options 增加：
  - `output_language: python`
  - `generate_options: qt_gui`

**改后功能**：
- 默认流图在工作流参数上更完整明确；
- 与新的 workflow 机制、GUI/生成链路更一致，减少空值导致的不确定行为。

---

### 10) `grc/gui/AgentPanel.py`（新增，核心）
**文件作用**：GTK 右侧 Agent 聊天面板。  
**本次改动**：
- 新增聊天 UI（历史区 + 输入框 + 发送按钮）
- 新增 `open_flow_graph` 信号：建图成功后通知主窗口打开 `.grc`
- 新增后台线程执行建图，避免 GTK 主线程卡死
- 支持简单多轮历史（`self._history`）

**改后功能**：
- 用户可在 GUI 里直接“用自然语言提需求”，系统自动生成并载入流图；
- 具备基本交互体验与非阻塞执行。

---

### 11) `grc/gui/Application.py`
**文件作用**：GTK 应用生命周期入口。  
**本次改动**：
- `Gtk.Application` 增加 `application_id="org.gnuradio.grc"` 和 `NON_UNIQUE` flag
- `do_activate` 中显式 `show_all()` + `present()`

**改后功能**：
- 重点修复 macOS/Quartz 下窗口不显示或进程过早退出问题；
- 应用激活与窗口映射更稳定。

---

### 12) `grc/gui/MainWindow.py`
**文件作用**：主窗口布局与面板可见性控制。  
**本次改动**：
- 引入 `AgentPanel`
- 右侧面板改为 `Gtk.Stack + StackSwitcher`：
  - `Core`（原块树）
  - `Agent`（新聊天面板）
- 增加 `_on_agent_open_flow_graph`：接收 Agent 生成路径后开新页
- 更新 panel 可见性逻辑：从 `btwin` 单体切换为 `right_top/stack` 容器控制

**改后功能**：
- 右侧栏支持“块树/Agent”标签切换，类似 IDE 侧边栏；
- Agent 生成结果可一键回灌到编辑器页签。

---

### 13) `grc/main.py`
**文件作用**：程序启动入口（GTK/QT）。  
**本次改动**（GTK 路径）：
- 引入 `from .agent import env`
- 在 build library 前调用 `env.bridge_package_name()`
- `platform.build_library()` 改为 `platform.build_library(env.block_paths())`
- 增加 `if __name__ == '__main__': main()`

**改后功能**：
- 解决“源码树 GRC + conda GNU Radio 运行时”混搭场景下的关键桥接：
  1. `grc` 包映射为 `gnuradio.grc`，使 workflow 生成模块可导入  
  2. 把源码树 `grc/blocks` 加入 block 搜索路径，确保 `*.workflow.yml` 可加载
- 启动入口更标准，可直接脚本方式执行。

---

# 这次分支改动的“整体功能结论”
**“把 Agent 能力正式接入 GRC GTK GUI 的端到端落地”**，并修复了 workflow/启动稳定性问题：

1. **能力新增**：自然语言 → LLM 生成 `.grc` → 校验保存 → GUI 自动打开  
2. **架构整理**：`agent-backend` 重命名归并到 `grc/agent`  
3. **稳定性修复**：workflow 匹配、默认 flowgraph 参数、macOS GTK 显示生命周期  
4. **运行时兼容**：source-tree + conda 混合环境桥接



# 260818 cindysha

> 启动方式（gnuradio 虚拟环境下）：
> `PYTHONPATH=$PWD python -m grc.main --gtk` 拉起 GTK 窗口，右侧 Agent 面板可交互。

## agent 目前执行的任务是什么？
**不是二选一，而是一条完整闭环**：自然语言意图 → 建图（生成可运行 `.grc`）→ 仿真（出星座/频谱/眼图）→ 按指标调参。

- **本质是“波形设计 + 仿真验证”**，生成的 `.grc` 使用仿真信源/信道（如 AWGN），可编译成 Python 跑起来看指标。
- **不是直接驱动 SDR 硬件发射射频**：要真发射需把流图里的仿真 sink 换成 USRP/osmocom 等硬件块，agent 默认走仿真路径。
- 两种工作模式（GUI 面板可切换）：
  1. **一句话直出（baseline）**：`build_flow_graph_from_text` 直接 LLM 产 `.grc`，只建图不主动仿真；
  2. **多轮协商 Agent（今天接进 GUI）**：走 planner 五阶段 `INTENT → PROPOSE → BUILD → SIMULATE → TUNE → DONE`，可建图、可仿真、可按 EVM/Eb·N0 调参。

---

### 1) `grc/agent/tools/skill_tools.py`（新增，核心）
**文件作用**：宏工具层——把 `skills` 的编排能力（`design_link`/`debug_by_metric`）暴露给 LLM function-calling。  
**本次改动**：
- 用 `@tool(group="macro")` 把 `design_link`（一步按配方建图）、`debug_by_metric`（一步按指标诊断）注册为“宏工具”
- 通过 `ctx.extra["profile"]` 桥接用户专业度画像，使宏工具也能按档位渲染叙述（narrative）

**改后功能**：
- LLM 在 BUILD 阶段可一步 `design_link` 建图、在 TUNE 阶段一步 `debug_by_metric` 诊断，无需逐块手工搭建。

---

### 2) `grc/agent/tools/registry.py`
**文件作用**：工具注册表（`@tool` 装饰器 + JSON-Schema + 调度入口）。  
**本次改动**：
- `_TOOL_MODULES` 新增 `"skill_tools"`，使宏工具在 `load_all()` 时被自动发现并注册。

**改后功能**：
- 宏工具正式进入 `macro` 分组，可导出 function-calling schema 供模型调用。

---

### 3) `grc/agent/core/planner.py`
**文件作用**：五阶段状态机，约束每个阶段允许调用的工具分组。  
**本次改动**：
- `_ALLOWED_GROUPS` 在 `BUILD` 与 `TUNE` 阶段放开 `macro` 组

**改后功能**：
- 让 LLM 在建图/调参阶段可合法调用宏工具，同时仍保持 ReAct 不越权乱调其它阶段工具。

---

### 4) `grc/agent/core/agent.py`
**文件作用**：Agent 编排内核（多轮 `step` 主循环）。  
**本次改动**：
- 每轮把 `self.ctx.profile` 注入 `tool_ctx.extra["profile"]`，使宏工具也能按档位渲染叙述（创新 B 贯穿到 function-calling 路径）
- 增强 `_merge_artifacts`：把宏工具嵌套的 `artifacts`（`grc_path`/`constellation_png`/`spectrum_png`/`eye_png`）与 `metrics` 上浮到顶层

**改后功能**：
- GUI 与实验埋点都能直接从顶层 `artifacts` 拿到 `.grc` 路径与产物图；专业度画像对宏工具生效。

---

### 5) `grc/agent/experiments/__init__.py` + `grc/agent/experiments/ablation.py`（新增）
**文件作用**：CHI 实验脚手架——用 `Session` 埋点 + `FlowGraphStore` 复用，跑量化消融。  
**本次改动**：新增三条量化实验
- **E1 自适应 vs 固定档位**：同一段“专业度渐变”对话，对比 `adaptive` 开/关下的档位轨迹（断言“非降且末档 > 首档”）
- **E2 三档表达差异化**：同一 `design_link` 结果在 novice/student/expert 三档下叙述的差异度（0.68~0.82）
- **E3 经验复用**：`FlowGraphStore` 记住一次建图后，相似意图能否召回同一配方省往返

**改后功能**：
- 为论文提供可复现、可量化的消融证据；三条实验全部 PASS。

---

### 6) `grc/gui/AgentPanel.py`（重写，核心）
**文件作用**：GTK 右侧 Agent 聊天面板。  
**本次改动**：
- 从旧“一句话直出”升级为**多轮协商**：持有 `Agent` 实例逐轮 `agent.step`，回显按档位渲染的叙述
- **内联展示产物图**：把 `artifacts` 里的星座/频谱/眼图缩略显示在面板内
- 产出 `.grc` 时 emit `open_flow_graph` 信号让主窗口载入画布
- 新增**专业度档位下拉**（自适应/小白/学生/专家，体现创新 B：可手动钉档或让其自适应）
- 保留 **baseline 开关**（“一句话直出”）作为论文对照
- 后台线程执行、非阻塞 GTK 主循环

**改后功能**：
- 用户在 GUI 里即可完成“提需求 → 看叙述 → 看产物图 → 一键载入流图”的多轮闭环，并可切换 baseline 做对照。

---

# 这次改动的“整体功能结论”
**“把 skills 编排能力升级为宏工具并贯通到 LLM/GUI，形成可量化验证的多轮协商闭环”**：

1. **能力升级**：宏工具（`design_link`/`debug_by_metric`）接入 registry/planner/agent，LLM 可一步建图/一步诊断
2. **GUI 升级**：AgentPanel 走多轮协商 + 内联产物图 + 档位控件，保留 baseline 对照
3. **实验支撑**：新增 ablation E1/E2/E3 三条量化消融，全部 PASS
4. **产物贯通**：宏工具嵌套产物/指标上浮，GUI 与埋点统一可读



# 260818 cindysha（修复：意图阶段空转到"工具调用上限"）

## 现象
小白档下输入"给我一个最简单的例子，一个正弦单音加点噪声，看看频谱"，Agent 回复"（已达工具调用步数上限）"。

> 附带的两条终端日志均无害：
> - `Gtk-WARNING: Could not load a pixbuf ... bullet-symbolic.svg` —— conda 环境 GTK/Adwaita 图标主题缺 pixbuf loader，纯渲染警告。
> - `TSM AdjustCapsLockLED... / IMKCFRunLoopWakeUpReliable` —— macOS 输入法框架（TSM/IMK）系统级日志，与本程序无关。

## 根因（编排 bug，非网络/环境问题）
多轮 Agent 首阶段是 **INTENT（意图澄清）**，本该纯自然语言复述需求、请用户确认，**不需要调工具**。但原逻辑给该阶段挂了 `knowledge` 组检索工具（`search_blocks`）并强制走 function-calling 循环。GLM 习惯性反复调检索（连调 8 次）却从不产出文本，`_run_toolcall_loop` 跑满 `max_tool_steps` 后兜底抛出"（已达工具调用步数上限）"这句生硬文案。

---

### `grc/agent/core/agent.py`（本次唯一改动文件，核心）
**文件作用**：Agent 编排内核（多轮 `step` 主循环、function-calling / 文本 ReAct 两条执行路径、system prompt 组装）。  
**本次改动**：
- **意图/方案阶段不再传工具**：`_step_with_llm` 中判断 `stage in (INTENT, PROPOSE)` 时直接调 `_force_text_reply` 一次性出自然语言回复，从机制上杜绝"只有检索工具时反复空转"（最关键的一处）。
- **新增 `_force_text_reply`**：不带 `tools` 请求一次、强制模型给自然语言回复。同时服务两个场景：(1) 纯协商阶段的一次性直出；(2) tools 循环耗尽后的兜底总结。用 `list(messages)` 局部副本追加一条 system 引导语，**不污染调用方 messages**（避免连续两条 user）。
- **循环耗尽兜底更健壮**：`_run_toolcall_loop` 与 `_run_react_text` 跑满步数时，不再返回死板的"（已达工具调用步数上限）"，改为调 `_force_text_reply` 逼模型基于已有 observation 给一句真回复。
- **`max_tool_steps` 6 → 8**：给 BUILD/SIMULATE 等真需要多步工具的阶段留余量。
- **system prompt 分阶段职责化**：`_system_prompt` 按 `stage` 注入分阶段说明（intent/propose 以自然语言协商为主、工具可选勿空转；build/simulate/tune 才真正调工具），并强调"每轮最终都必须给出一句面向用户的自然语言回复"。

**改后功能**：
- INTENT/PROPOSE 阶段 **0 次工具调用**、直接出贴合档位的自然语言回复，`needs_confirmation=True` 正常。
- 即便在 BUILD/SIMULATE 阶段跑满步数，也能兜底拿到一句真回复而非"上限"文案。
- 彻底消除"（已达工具调用步数上限）"。

---

## 验证
实跑复现场景（小白档 + 那句需求）：
- INTENT 阶段：`TOOLS: []`，直接出小白档意图复述，`needs_confirm=True`。
- 回复"对，就是这个意思" → 进入 PROPOSE：同样 `TOOLS: []`，直接给出"三步走"方案。
- 不再出现"（已达工具调用步数上限）"。

---

# 这次改动的"整体功能结论"
**"修复多轮 Agent 意图/方案阶段的工具空转 bug，让纯协商阶段稳定直出自然语言回复"**：

1. **机制修复**：INTENT/PROPOSE 不传工具、一次性直出，杜绝空转（根治）
2. **兜底加固**：新增 `_force_text_reply`，循环耗尽也能出真回复而非生硬文案
3. **提示优化**：system prompt 分阶段职责 + 强制每轮产出自然语言
4. **余量调整**：`max_tool_steps` 6 → 8，兼顾多步工具阶段

